# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 04 · Observed-origin feature laboratory
## NFL trajectory research · Round 1

**Question:** Can task-conditioned arrival features and additional earlier observation origins improve longer-horizon forecasts?

This notebook constructs real training-side examples; it does not fit a model. The next notebook runs a small attributable feature diagnostic. Your successful workspace sync is not repeated. Data, existing models, Git branches, and Python packages are not modified.

**Use the `NFL Trajectory (Python 3.11)` kernel. Run cells in order.** Each worker has a hard time cap, a 15-second heartbeat, and per-play checkpoints. Do not run another notebook stage concurrently.

Prepared views at **0, 5, 10 and 20 frames earlier** keep the original post-pass targets. Withheld pre-pass coordinates become labels only, never features. The ball landing point, roles, prediction flags and horizon remain task-supplied information; these pseudo-origins are a competition-training device, not a claim of live earlier-time information availability.

The previous report contains no new Kaggle result. The historical target remains **0.46340**; the diagnostics below are not directly comparable to that leaderboard score.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import pandas as pd

KIT = Path.cwd()
if not (KIT / "run_round.py").is_file():
    KIT = Path("/home/sagemaker-user/nfl_feature_round1")
if not (KIT / "run_round.py").is_file():
    raise FileNotFoundError("Open this notebook from the extracted nfl_feature_round1 folder.")
sys.path.insert(0, str(KIT))
from visuals import show_save
OUT = Path("/home/sagemaker-user/nfl-feature-round1-results")
FIGURES = OUT / "figures"

def run_stage(command, *arguments, seconds=600):
    cmd = [sys.executable, str(KIT / "run_round.py"), command, *map(str, arguments), "--seconds", str(seconds)]
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=KIT, check=True, timeout=seconds + 30)

print("Kernel:", sys.executable)
print("Outputs:", OUT)


## 1 · Why this research direction
The plot is calculated from your uploaded **saved velocity-experiment receipt**, not from a newly trained model. It weights each horizon bin by its row count and squared coordinate RMSE. Historical outer-validation diagnostics motivate the hypothesis; they are not used to tune this round's thresholds or fitting parameters.

In [ ]:
from visuals import old_error_slices, error_mass_figure
prior = json.loads((KIT / "evidence" / "previous_workspace_receipts.json").read_text())
slices = old_error_slices(prior)
late = slices["value"] > 1
print(f"After first second: {slices.loc[late, 'row_share'].sum():.2%} of rows; "
      f"{slices.loc[late, 'squared_error_share'].sum():.2%} of squared error.")
display(slices[["value", "rows", "coordinate_rmse_yards", "row_share", "squared_error_share"]])
show_save(error_mass_figure(slices), FIGURES / "recovered_error_mass.html")

## 2 · Narrow input check
Verify the reviewed revision and exact existing private cache before reading its split/keys. No package installation, broad re-download, new cloud job or Git synchronization occurs. A changed revision/cache is a stop requiring diagnosis, not a reason to reset the workspace.

In [ ]:
run_stage("preflight", seconds=120)
display(pd.Series(json.loads((OUT / "preflight.json").read_text()), name="verified"))

## 3 · Construct 32 plays first
This is a new **raw-to-shifted-origin** smoke, not a repeat of the earlier temporal-edge smoke. It tests the changed information boundary and target reconstruction. No fitted model is created. All original forecast rows must remain present at offset zero. Ineligible shifted views are counted with reasons, not silently treated as successful examples.

In [ ]:
run_stage("prepare", "--plays", 32, "--label", "smoke", seconds=180)
smoke = json.loads((OUT / "smoke" / "preparation_summary.json").read_text())
assert smoke["status"] == "feature_dataset_ready"
display(pd.DataFrame(smoke["offsets"]).T)
from visuals import origins_figure
show_save(origins_figure(smoke), FIGURES / "smoke_origin_targets.html")

## 4 · Build the bounded research sample
This selects **256 plays without looking at outcomes**, round-robin across the original training games, and checkpoints every play at four origins. It does not consume the existing 41-game validation partition or reserved holdout for training/selection.

The cap is **600 seconds**, not a predicted runtime. A timeout preserves completed play artifacts. Do not increase the cap or change the scientific recipe on the basis of validation results. Report an error; do not repeatedly retry an unchanged failing stage.

In [ ]:
run_stage("prepare", "--plays", 256, "--label", "research", seconds=600)
research = json.loads((OUT / "research" / "preparation_summary.json").read_text())
assert research["status"] == "feature_dataset_ready"
display(pd.Series({k:v for k,v in research.items() if k != "offsets"}))
show_save(origins_figure(research), FIGURES / "research_origin_targets.html")

## 5 · Inspect support, not just column count
The diagnostic has **72 control columns** and **24 role-conditioned arrival candidates**. These are physical displacement-response hypotheses, not 24 proven new signals. Some columns will be structurally zero for roles that are not scored, and training-fold screening drops constant columns.

Recent history, receiver/passer geometry and orientation are already in the control. The added family concerns velocity/acceleration required to reach the supplied landing point, and receiver-relative motion. It overlaps established domain concepts in the repository; the materially new research treatment is the carefully reconstructed **earlier-origin training policy**.

Read the feature dictionary and protocol before interpreting an apparent win.

In [ ]:
from run_round import load_dataset
from origin_features import ALL_NAMES
from visuals import horizon_figure, feature_support_figure
data = load_dataset(OUT / "research")
assert np.isfinite(data["X"]).all()
show_save(horizon_figure(data), FIGURES / "training_horizon_support.html")
fig, support = feature_support_figure(data, ALL_NAMES)
show_save(fig, FIGURES / "arrival_activation.html")
support.to_csv(OUT / "research" / "feature_support.csv", index=False)
display(support.groupby("origin_frames")[["finite_fraction", "nonzero_fraction"]].mean())

## 6 · Milestone review
**Completed only when the preceding cells succeed:** input-contract verification, raw-data origin reconstruction, original-row preservation, training-only feature materialization, and support diagnostics.

**Not completed here:** measured feature value, the established neural-model experiment, independent forward replay of the previous velocity model, a Kaggle submission, or proof of a leaderboard improvement.

Next open **`05_feature_ablation.ipynb`**. It runs three fixed-ridge arms on three chronological training-side folds, for at most nine small fits. If preparation failed, export the available receipt instead of starting the screen.

In [ ]:
run_stage("report", seconds=60)
print("Preparation checkpoint:", OUT / "research" / "preparation_summary.json")
print("Return report:", OUT / "nfl_feature_round1_report.zip")